# Visual-Doc Assistant — Phase 3
## Gemini Answer Generation & Streamlit UI

**Objective:** Take a user's natural-language query, retrieve the most relevant page(s) from the ChromaDB index built in Phase 2, pass those page images + the query to Gemini for multimodal answer generation, and expose the whole pipeline through a Streamlit UI — completing the end-to-end Visual-Doc Assistant.

---
### Phase Context
```
Phase 1  →  PDF → Images + ChromaDB initialised (0 embeddings)
Phase 2  →  Images → ColPali → 128-dim Embeddings → ChromaDB
           +Query → ColPali → 128-dim Query Vector → Sanity Check
Phase 3  →  Query → ColPali embed_query() → ChromaDB retrieval → Top-K Pages
           → Gemini multimodal → Answer → Streamlit UI              THIS NOTEBOOK
```

> **GPU required:** `Runtime → Change runtime type → T4 GPU`. ColPali still needs to embed the *live* query at question-time using the same model from Phase 2, so this notebook reloads it.

> **Heads-up on the Phase 2 notebook, before you run Step 1:** `Phase2_Implementation.ipynb`'s install cell pins `colpali-engine==0.3.7`, which isn't published on PyPI, and its Drive wheel cache has no logic to detect stale versions. Phase 2's results are all verified end-to-end, so this was evidently patched by hand in the live session at the time, but the fix never made it back into the saved notebook. Step 1 below installs `colpali-engine==0.3.17` instead (the current release). Worth back-porting this same install cell into Phase 2 next time you touch it, so a cold restart of that notebook doesn't break.
>
> **Second correction, if you hit a `NotImplementedError: Cannot copy out of meta tensor` on Step 2:** an earlier version of this Step 1 cell installed `peft` and `torchao` with no upper version bound. `colpali-engine`'s own published metadata pins `peft>=0.18.0,<0.20.0` and `transformers>=5.3.0,<6.0.0` specifically — its LoRA adapter key-remapping fix for this exact checkpoint's `custom_text_proj` layer (landed in colpali-engine v0.3.14) was built and tested against that range, and an unconstrained `peft` install can resolve outside it, breaking that remapping and leaving part of the model uninitialized on the "meta" device. Step 1 now pins `peft`/`transformers` to colpali-engine's own declared ranges instead of leaving them open, and `torchao` is dropped entirely — it isn't actually a colpali-engine dependency and nothing here imports it. **If you already hit this error, a plain "Restart runtime" won't fix it** — Colab keeps pip-installed packages across a kernel restart within the same VM. Use **Runtime → Disconnect and delete runtime**, reconnect, and re-run from Step 1 below.


## Step 1 — Mount Drive & Reinstall Dependencies

Same Drive-cached-wheel folder as Phase 2 (`Visual-Doc/deps/`, so packages persist across ephemeral Colab sessions), but installed with `--find-links` rather than `--no-index`: the cache is used when it's valid, but pip can still fall back to PyPI to correct anything stale or missing, instead of silently installing whatever happens to already be sitting in the cache or the environment.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import glob, os
for pattern in ['pillow*', 'torchao*']:
    for f in glob.glob(f'/content/drive/MyDrive/Visual-Doc/deps/{pattern}'):
        os.remove(f)
        print('removed', f)

removed /content/drive/MyDrive/Visual-Doc/deps/pillow-11.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
removed /content/drive/MyDrive/Visual-Doc/deps/torchao-0.18.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl


In [3]:
import sys, os, subprocess

WHEELS_DIR = '/content/drive/MyDrive/Visual-Doc/deps'
os.makedirs(WHEELS_DIR, exist_ok=True)

# colpali-engine==0.3.7 (as pinned in Phase2_Implementation.ipynb) is not on PyPI — using
# the current release instead. Its own published metadata pins peft>=0.18.0,<0.20.0 and
# transformers>=5.3.0,<6.0.0 (confirmed against PyPI + the colpali-engine v0.3.14 changelog,
# which is where its LoRA adapter key-remapping fix for this checkpoint's custom_text_proj
# layer landed) — declaring them explicitly here, in the SAME install command as
# colpali-engine, so pip resolves all of them together instead of picking an incompatible
# version for one of them separately. torchao is dropped: it's not an actual colpali-engine
# dependency and nothing in this notebook imports it.
CORE = [
    'colpali-engine==0.3.17',
    'peft>=0.18.0,<0.20.0',
    'transformers>=5.3.0,<6.0.0',
    'chromadb',
    'Pillow<12.0.0',
    'torchao>=0.16.0',
]

def pip_run(cmd, label):
    """Run a pip command, print a clean one-line result."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    our = ['colpali', 'chromadb', 'pillow', 'peft', 'transformers']
    errs = [l for l in result.stderr.splitlines()
            if 'ERROR' in l and any(p in l.lower() for p in our)]
    if errs:
        print(f'  {label}:')
        for e in errs: print(f'   {e}')
    else:
        print(f' {label}')

# Populate/refresh the Drive wheel cache (cheap no-op on repeat runs once it's warm)
pip_run(
    [sys.executable, '-m', 'pip', 'download', '--dest', WHEELS_DIR, '--quiet'] + CORE,
    f'Wheels cached to {WHEELS_DIR}'
)

# Install with --find-links (prefer the Drive cache) but deliberately WITHOUT --no-index,
# and with --upgrade: pip can still reach PyPI to correct anything the cache is missing or
# that's the wrong version, rather than silently keeping a stale/incompatible package in
# place. --no-index (used in the first attempt at this cell) is what let an unconstrained
# peft install go unnoticed instead of being corrected here.
pip_run(
    [sys.executable, '-m', 'pip', 'install',
     '--find-links', WHEELS_DIR, '--upgrade', '--quiet', '--no-warn-conflicts'] + CORE,
    'Core packages installed'
)

print()
try:
    import colpali_engine, chromadb, transformers, peft
    print(f' colpali_engine : {getattr(colpali_engine, "__version__", "n/a")}')
    print(f' transformers   : {transformers.__version__}')
    print(f' peft           : {peft.__version__}')
    print(f' chromadb       : {chromadb.__version__}')
except ImportError as e:
    print(f' Import failed: {e}')

 Wheels cached to /content/drive/MyDrive/Visual-Doc/deps
 Core packages installed

 colpali_engine : n/a
 transformers   : 5.23.0
 peft           : 0.19.1
 chromadb       : 1.5.9


## Step 2 — Reload ColPali Vision-Language Model

Identical to Phase 2, Step 2. Needed here because retrieval at query-time requires encoding the user's live question into the same 128-dim space the stored page embeddings live in.

In [4]:
import torch
from PIL import Image
from IPython.display import display, Image as IPImage
from colpali_engine.models import ColPali, ColPaliProcessor

MODEL_NAME = 'vidore/colpali-v1.2'
EMBED_DIM  = 128
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device   : {DEVICE}')
print(f'Embed dim: {EMBED_DIM}')
if DEVICE == 'cpu':
    print('  No GPU detected — switch to T4 GPU runtime before continuing.')

print(f'\nLoading {MODEL_NAME} ...')
model = ColPali.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()

processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

print(f'\n ColPali loaded on {DEVICE} — ready to embed live queries.')

Device   : cuda
Embed dim: 128

Loading vidore/colpali-v1.2 ...



 ColPali loaded on cuda — ready to embed live queries.


In [5]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

adapter_path = hf_hub_download('vidore/colpali-v1.2', filename='adapter_model.safetensors')
adapter_state_dict = load_file(adapter_path)

target_dtype  = next(model.parameters()).dtype
target_device = next(model.parameters()).device

def remap_key(k):
    if k.startswith('base_model.model.'):
        k = k[len('base_model.model.'):]
    if k.endswith('lora_A.weight') or k.endswith('lora_B.weight'):
        k = k[:-len('.weight')] + '.default.weight'
    return k

remapped = {
    remap_key(k): v.to(dtype=target_dtype, device=target_device)
    for k, v in adapter_state_dict.items()
}

model.load_state_dict(remapped, strict=False)

# lora_B is zero-initialized by default — a nonzero norm here confirms the
# real trained weights made it in, not just a freshly-initialized adapter.
b_norm = model.custom_text_proj.lora_B.default.weight.norm().item()
print(f'custom_text_proj lora_B weight norm: {b_norm:.4f}')
print('Loaded correctly.' if b_norm > 0 else 'Still zero — not loaded.')

custom_text_proj lora_B weight norm: 0.2158
Loaded correctly.


## Step 3 — Reconnect to Phase 2's ChromaDB Collection

No re-embedding here — Phase 2 already populated all 26 pages. This just reconnects to that existing collection on Drive and verifies it's intact.

In [6]:
import os
import numpy as np
import chromadb

IMAGE_FOLDER = '/content/drive/MyDrive/Visual-Doc/processed_images'
DB_PATH      = '/content/drive/MyDrive/Visual-Doc/chroma_db_storage'

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection(
    name='visual_doc_collection',
    metadata={'hnsw:space': 'cosine'}
)

print(f'Collection "{collection.name}" — {collection.count()} embeddings stored.')
assert collection.count() > 0, "No embeddings found — run Phase 2 first."
print('Phase 2 index verified. Ready for retrieval.')

Collection "visual_doc_collection" — 521 embeddings stored.
Phase 2 index verified. Ready for retrieval.


## Step 4 — `embed_query()` (unchanged from Phase 2)

Reproduced here rather than imported, so this notebook is self-contained and runnable on its own in a fresh runtime. Kept byte-for-byte identical to Phase 2 so retrieval behaves exactly as already verified there.

In [7]:
@torch.no_grad()
def embed_query(query_text: str) -> np.ndarray:
    """
    Generate a 128-dim embedding for a text query using ColPali's text encoder.
    Same model and projection layer as Phase 2's embed_page_image — query and
    image embeddings live in the same semantic space.
    """
    inputs           = processor.process_queries([query_text]).to(DEVICE)
    token_embeddings = model(**inputs)                          # (1, N_tokens, 128)
    pooled           = token_embeddings.mean(dim=1).squeeze(0)  # (128,)
    return pooled.cpu().float().numpy()

## Step 5 — Retrieval Function

Adapted from Phase 2's `retrieval_sanity_check` — same embed → cosine search logic, but returns structured results instead of just printing/displaying, so Gemini prompt construction and the Streamlit UI can consume them directly.

In [8]:
def retrieve_top_k_pages(query_text: str, k: int = 3) -> list:
    """
    Embed the query with ColPali and return the top-k most similar pages
    from the Phase 2 ChromaDB index.
    """
    query_embedding = embed_query(query_text)

    results = collection.query(
        query_embeddings = [query_embedding.tolist()],
        n_results        = k,
        include          = ['metadatas', 'distances'],
    )

    pages = []
    for meta, dist in zip(results['metadatas'][0], results['distances'][0]):
        pages.append({
            'page_number': meta['page_number'],
            'image_path':  meta['image_path'],
            'similarity':  1 - dist,   # ChromaDB returns cosine distance
        })
    return pages

## Step 6 — Install & Configure the Gemini API

Uses the `google-genai` SDK and its Interactions API (the current recommended way to call Gemini — the older `generateContent`-only style and the deprecated `google-generativeai` package are both superseded by this). Get a free API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).

In [9]:
!pip install -q -U google-genai

import os
from getpass import getpass
from google import genai

if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = getpass('Enter your Gemini API key (from https://aistudio.google.com/apikey): ')

# Named gemini_client (not "client") — "client" is already the ChromaDB PersistentClient from Step 3
gemini_client = genai.Client()
GEMINI_MODEL  = 'gemini-3.6-flash'   # strong multimodal quality/latency balance
                                     # swap to 'gemini-3.5-flash-lite' for a faster/cheaper option

print(f' Gemini client ready — using {GEMINI_MODEL}')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.4.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
Enter your Gemini API key (from https://aistudio.google.com/apikey): ··········
 Gemini client ready — using gemini-3.6-flash


## Step 7 — Multimodal Prompt Construction & Answer Generation

Retrieved page images are sent inline as base64 (well within the Interactions API's 20MB inline-request limit for a handful of 300-DPI page PNGs — see Phase 1's ~200KB–1MB-per-page estimate). A system instruction grounds Gemini in *only* the retrieved pages, so it doesn't fall back on outside knowledge.

In [10]:
import base64

SYSTEM_INSTRUCTION = (
    "You are the Visual-Doc Assistant, a technical documentation Q&A system. "
    "Answer the user's question using ONLY the page image(s) provided below — "
    "do not use outside knowledge. Reference page numbers when relevant. "
    "If the provided pages don't contain the answer, say so directly instead of guessing."
)

def build_multimodal_input(query_text: str, pages: list) -> list:
    """Builds the Gemini `input` list: query text first, then each retrieved page image inline."""
    content = [{'type': 'text', 'text': f'Question: {query_text}'}]
    for p in pages:
        with open(p['image_path'], 'rb') as f:
            image_bytes = f.read()
        content.append({
            'type':      'image',
            'data':      base64.b64encode(image_bytes).decode('utf-8'),
            'mime_type': 'image/png',
        })
    return content

def generate_answer(query_text: str, pages: list) -> str:
    """Send the query + retrieved page images to Gemini and return its text answer."""
    try:
        interaction = gemini_client.interactions.create(
            model              = GEMINI_MODEL,
            input              = build_multimodal_input(query_text, pages),
            system_instruction = SYSTEM_INSTRUCTION,
        )
        return interaction.output_text
    except Exception as e:
        return f" Gemini request failed: {e}"


## Step 8 — End-to-End QA Pipeline & Sanity Checks

Same two queries used in Phase 2's retrieval sanity check — now with a real generated answer attached, proving the full Phase 3 loop (embed → retrieve → generate → display) end to end.

In [11]:
# def ask(query_text: str, k: int = 3, show_pages: bool = True):
#     """
#     Full Phase 3 pipeline: embed the query, retrieve top-k pages from ChromaDB,
#     send them + the query to Gemini, and print the answer alongside source pages.
#     """
#     print(f'Query: "{query_text}"\n')
#     pages = retrieve_top_k_pages(query_text, k=k)

#     print(f'Top-{k} retrieved pages:')
#     for p in pages:
#         print(f"  Page {p['page_number']:>2} — cosine similarity: {p['similarity']:.4f}")

#     answer = generate_answer(query_text, pages)
#     print('\nGemini answer:')
#     print(answer)

#     if show_pages:
#         print('\nSource pages:')
#         for p in pages:
#             img = Image.open(p['image_path']).convert('RGB')
#             img.thumbnail((400, 600))
#             display(img)

#     print()
#     return answer, pages


# # ── Sanity checks — same queries as Phase 2, now with real answer generation ──
# ask('What is the system architecture and pipeline diagram?', k=3)
# ask('What are the results and performance comparisons?', k=3)

In [12]:
!apt-get update -qq
!apt-get install -y poppler-utils -qq
!pip install pdf2image -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_24.02.0-1ubuntu9.11_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (24.02.0-1ubuntu9.11) over (24.02.0-1ubuntu9.10) ...
Preparing to unpack .../libpoppler-dev_24.02.0-1ubuntu9.11_amd64.deb ...
Unpacking libpoppler-dev:amd64 (24.02.0-1ubuntu9.11) over (24.02.0-1ubuntu9.10) ...
Preparing to unpack .../libpoppler134_24.02.0-1ubuntu9.11_amd64.deb ...
Unpacking libpoppler134:amd64 (24.02.0-1ubuntu9.11) over (24.02.0-1ubuntu9.10) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.11_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.11) ...
Setting up libpoppler134:amd64 (24.02.0-1ubuntu9.11) ...
Setting up poppler-

## Step 9 — Streamlit UI

Writes a standalone `app.py`. It reloads its own copy of everything above (a Streamlit app runs as a separate process, so it can't import the cells above directly) but the logic is identical, wrapped in `@st.cache_resource` so the ColPali model and ChromaDB connection load once per session rather than on every query.

In [13]:
%%writefile /content/app.py
"""
Visual-Doc Assistant — Phase 3 Streamlit UI
Run with: streamlit run /content/app.py   (see the next cell for launching it from Colab)
"""
import os
import base64

import streamlit as st
import torch
from PIL import Image
import chromadb
from colpali_engine.models import ColPali, ColPaliProcessor
from google import genai

# ── Config (mirrors the notebook above) ─────────────────────────────────────
MODEL_NAME   = 'vidore/colpali-v1.2'
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_FOLDER = '/content/drive/MyDrive/Visual-Doc/processed_images'
DB_PATH      = '/content/drive/MyDrive/Visual-Doc/chroma_db_storage'
GEMINI_MODEL = 'gemini-3.6-flash'

SYSTEM_INSTRUCTION = (
    "You are the Visual-Doc Assistant, a technical documentation Q&A system. "
    "Answer the user's question using ONLY the page image(s) provided below — "
    "do not use outside knowledge. Reference page numbers when relevant. "
    "If the provided pages don't contain the answer, say so directly instead of guessing."
)

st.set_page_config(page_title="Visual-Doc Assistant", page_icon="📄", layout="wide")

# ── Cached resources: load once per session, not once per query ────────────
@st.cache_resource(show_spinner="Loading ColPali model (first load only)...")
def load_colpali():
    model = ColPali.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16).eval().to(DEVICE)
    processor = ColPaliProcessor.from_pretrained(MODEL_NAME)
    return model, processor

@st.cache_resource(show_spinner="Connecting to ChromaDB...")
def load_collection():
    db_client = chromadb.PersistentClient(path=DB_PATH)
    return db_client.get_or_create_collection(
        name='visual_doc_collection', metadata={'hnsw:space': 'cosine'}
    )

model, processor = load_colpali()
collection = load_collection()

# ── Pipeline functions (identical logic to the Phase 3 notebook) ───────────
@torch.no_grad()
def embed_query(query_text):
    inputs = processor.process_queries([query_text]).to(DEVICE)
    token_embeddings = model(**inputs)
    pooled = token_embeddings.mean(dim=1).squeeze(0)
    return pooled.cpu().float().numpy()

def retrieve_top_k_pages(query_text, k=3):
    query_embedding = embed_query(query_text)
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k,
        include=['metadatas', 'distances'],
    )
    return [
        {'page_number': meta['page_number'], 'image_path': meta['image_path'], 'similarity': 1 - dist}
        for meta, dist in zip(results['metadatas'][0], results['distances'][0])
    ]

def build_multimodal_input(query_text, pages):
    content = [{'type': 'text', 'text': f'Question: {query_text}'}]
    for p in pages:
        with open(p['image_path'], 'rb') as f:
            image_bytes = f.read()
        content.append({
            'type': 'image',
            'data': base64.b64encode(image_bytes).decode('utf-8'),
            'mime_type': 'image/png',
        })
    return content

def generate_answer(gemini_client, query_text, pages):
    try:
        interaction = gemini_client.interactions.create(
            model=GEMINI_MODEL,
            input=build_multimodal_input(query_text, pages),
            system_instruction=SYSTEM_INSTRUCTION,
        )
        return interaction.output_text
    except Exception as e:
        return f"Gemini request failed: {e}"

# ── Sidebar ──────────────────────────────────────────────────────────────────
st.sidebar.title("Settings")
api_key = st.sidebar.text_input(
    "Gemini API key", type="password",
    value=os.environ.get("GEMINI_API_KEY", ""),
    help="Get one at aistudio.google.com/apikey",
)
top_k = st.sidebar.slider("Pages to retrieve (k)", min_value=1, max_value=5, value=3)
st.sidebar.caption(f"Index: {collection.count()} pages · Device: {DEVICE}")

# ── Main UI ──────────────────────────────────────────────────────────────────
st.title("Visual-Doc Assistant")
st.caption("Multimodal RAG over technical documents — ColPali retrieval + Gemini generation, no OCR.")

query = st.text_input("Ask a question about the document:", placeholder="e.g. What is the system architecture?")
ask_clicked = st.button("Ask", type="primary")

if ask_clicked:
    if not api_key:
        st.error("Please enter your Gemini API key in the sidebar.")
    elif not query.strip():
        st.warning("Please enter a question.")
    else:
        gemini_client = genai.Client(api_key=api_key)
        with st.spinner("Retrieving relevant pages..."):
            pages = retrieve_top_k_pages(query, k=top_k)
        with st.spinner("Generating answer with Gemini..."):
            answer = generate_answer(gemini_client, query, pages)

        st.subheader("Answer")
        st.write(answer)

        st.subheader("Source pages")
        cols = st.columns(len(pages))
        for col, p in zip(cols, pages):
            with col:
                img = Image.open(p['image_path']).convert('RGB')
                caption = f"Page {p['page_number']} · similarity {p['similarity']:.3f}"
                st.image(img, caption=caption, use_container_width=True)

Writing /content/app.py


In [14]:
import os
path = '/content/drive/MyDrive/Visual-Doc/app.py'
print("Exists:", os.path.exists(path))
if os.path.exists(path):
    content = open(path).read()
    print("Has device_map fix:", "device_map=DEVICE" in content)
    print("Has custom_text_proj patch:", "_patch_custom_text_proj" in content)

Exists: True
Has device_map fix: False
Has custom_text_proj patch: False


## Step 9B — Upgrade: Multi-PDF Upload, Document Library & Chat UI

Step 9 above writes an `app.py` that can only answer questions about the one
document already embedded by `Runner.ipynb` (Phase 1) and
`Phase2_Implementation.ipynb` (Phase 2) — the 26-page ColPali paper. Adding a
different PDF meant re-running those two notebooks by hand for every new
document, which was flagged as an unimplemented stretch goal in this
notebook's own Phase 3 Summary below ("Uploading a new PDF through the UI").

This cell overwrites that same `app.py` (same path Step 10 already launches,
so **no changes needed below**) with a version that wires Phase 1's PDF→image
step and Phase 2's embedding step directly into the Streamlit page itself:

- **Sidebar file uploader** (`accept_multiple_files=True`) — add any number of
  PDFs straight from the browser, with a live per-page indexing progress bar
- **Document Library panel** — every indexed PDF listed with its page count,
  a 🗑️ button to remove one document, and a "Clear ALL" reset
- **Content-hash IDs** — re-uploading an identical file is detected and
  skipped rather than re-embedded (same resume-safe pattern as Phase 2's
  batch-embedding loop); two different files with the same name don't collide
- **Cross-document retrieval** — queries search your whole library by
  default, or a chosen subset via a multiselect
- **Running chat history** (`st.chat_message` / `st.chat_input`) instead of
  only the last answer, with each turn keeping its own source-page gallery

`embed_query`, `retrieve_top_k_pages`'s core logic, `build_multimodal_input`,
`generate_answer`, and `SYSTEM_INSTRUCTION` are all unchanged from Steps 4–7
above — only extended to loop over many documents instead of one.

> **Your existing 26-page index is not replaced.** It will simply appear in
> the new sidebar's Document Library alongside anything else you upload. Phase
> 2's records predate the `doc_id_prefix` field the library/delete/filter
> controls key off of, so a one-time `migrate_legacy_metadata()` backfills it
> for those existing rows the first time this app runs — it's a no-op on
> every run after that.


In [15]:
%%writefile /content/drive/MyDrive/Visual-Doc/app.py
"""
Visual-Doc Assistant — Phase 3, Step 9B: Multi-PDF Interactive UI
====================================================================
Upgrades Step 9's app.py so ANY NUMBER of PDFs can be added directly from
the web page, instead of requiring Runner.ipynb + Phase2_Implementation.ipynb
to be re-run by hand for every new document. This wires Phase 1's PDF -> image
step and Phase 2's ColPali embedding step directly into the Streamlit UI --
exactly the stretch goal flagged as "not in scope" at the end of this
notebook's original Step 9/Phase 3 Summary.

Writes to the SAME path Step 10 already launches
(/content/drive/MyDrive/Visual-Doc/app.py), so Step 10 below needs no changes.

Retrieval/generation logic (embed_query, retrieve_top_k_pages,
build_multimodal_input, generate_answer, SYSTEM_INSTRUCTION) is kept
functionally identical to Step 4-7 above -- only extended with multi-document
support. Your existing 26-page ColPali index from Phase 2 is NOT replaced;
it will simply appear in the new sidebar's Document Library like any other
indexed PDF (a one-time metadata migration below adds the doc_id_prefix
field Phase 2's records predate, so the library/delete/filter controls work
on it too).
"""
import os
import shutil
import base64
import hashlib
import time

import streamlit as st
import torch
from PIL import Image
import chromadb
from pdf2image import convert_from_bytes
from colpali_engine.models import ColPali, ColPaliProcessor
from google import genai

# ============================================================================
# Config -- same Drive paths as Phases 1-3, so this shares the existing index
# ============================================================================
MODEL_NAME   = "vidore/colpali-v1.2"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_FOLDER = "/content/drive/MyDrive/Visual-Doc/processed_images"
DB_PATH      = "/content/drive/MyDrive/Visual-Doc/chroma_db_storage"

GEMINI_MODEL = "gemini-3.6-flash"
PDF_DPI      = 300   # Same as Phase 1

os.makedirs(IMAGE_FOLDER, exist_ok=True)
os.makedirs(DB_PATH, exist_ok=True)

SYSTEM_INSTRUCTION = (
    "You are the Visual-Doc Assistant, a technical documentation Q&A system. "
    "Answer the user's question using ONLY the page image(s) provided below — "
    "do not use outside knowledge. Reference the source document and page "
    "number when relevant. If the provided pages don't contain the answer, "
    "say so directly instead of guessing."
)

st.set_page_config(page_title="Visual-Doc Assistant", page_icon="📄", layout="wide")


# ============================================================================
# Cached resources -- load once per session, not once per query (Step 9 pattern)
# ============================================================================
'''@st.cache_resource(show_spinner="Loading ColPali model (first load only)...")
def load_colpali():
    model = ColPali.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16).eval().to(DEVICE)
    processor = ColPaliProcessor.from_pretrained(MODEL_NAME)
    return model, processor'''
@st.cache_resource(show_spinner="Loading ColPali model (first load only)...")
def load_colpali():
    model = ColPali.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="cuda:0",
    ).eval()

    processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

    return model, processor

@st.cache_resource(show_spinner="Connecting to ChromaDB...")
def load_collection():
    db_client = chromadb.PersistentClient(path=DB_PATH)
    return db_client.get_or_create_collection(
        name="visual_doc_collection", metadata={"hnsw:space": "cosine"}
    )


model, processor = load_colpali()
collection = load_collection()


@st.cache_resource(show_spinner="Checking index for legacy records...")
def migrate_legacy_metadata():
    """
    One-time backfill: Phase 2's original 26-page index was written before
    doc_id_prefix existed in the metadata schema (see Phase2_Implementation.ipynb,
    Step 5). This assigns those legacy records a doc_id_prefix derived from their
    doc_name, so the new sidebar's delete/filter controls work on them too.
    Safe to run every session -- it's a no-op once migrated.
    """
    if collection.count() == 0:
        return 0
    all_data = collection.get(include=["metadatas"])
    updates_ids, updates_meta = [], []
    for doc_id, meta in zip(all_data["ids"], all_data["metadatas"]):
        if "doc_id_prefix" not in meta:
            legacy_prefix = f"{meta.get('doc_name', 'legacy')}__legacy"
            new_meta = dict(meta)
            new_meta["doc_id_prefix"] = legacy_prefix
            updates_ids.append(doc_id)
            updates_meta.append(new_meta)
    if updates_ids:
        collection.update(ids=updates_ids, metadatas=updates_meta)
    return len(updates_ids)


migrated_count = migrate_legacy_metadata()


# ============================================================================
# Embedding functions -- identical logic to Steps 4/6 above
# ============================================================================
@torch.no_grad()
def embed_page_image(image: Image.Image):
    inputs = processor.process_images([image]).to(DEVICE)
    patch_embeddings = model(**inputs)                          # (1, N_patches, 128)
    pooled = patch_embeddings.mean(dim=1).squeeze(0)             # (128,)
    return pooled.cpu().float().numpy()


@torch.no_grad()
def embed_query(query_text: str):
    inputs = processor.process_queries([query_text]).to(DEVICE)
    token_embeddings = model(**inputs)                           # (1, N_tokens, 128)
    pooled = token_embeddings.mean(dim=1).squeeze(0)              # (128,)
    return pooled.cpu().float().numpy()


# ============================================================================
# NEW -- PDF ingestion pipeline wired into the UI (Phase 1 + Phase 2, on demand)
# ============================================================================
def _file_hash(file_bytes: bytes) -> str:
    """Short content hash so re-uploading the same file is recognized and
    doesn't get re-embedded, and two different files with the same name
    don't collide."""
    return hashlib.md5(file_bytes).hexdigest()[:10]


def index_pdf(uploaded_file, progress_bar=None) -> dict:
    """
    Convert an uploaded PDF into page images (Phase 1 logic) and embed +
    store every page in ChromaDB (Phase 2 logic). Resume-safe: pages already
    present in the collection (matched by content-hash + page number) are
    skipped, exactly like Phase 2's batch-embedding loop.
    """
    file_bytes = uploaded_file.getvalue()
    doc_hash = _file_hash(file_bytes)
    doc_name = os.path.splitext(uploaded_file.name)[0]
    doc_id_prefix = f"{doc_name}__{doc_hash}"

    doc_image_folder = os.path.join(IMAGE_FOLDER, doc_id_prefix)
    os.makedirs(doc_image_folder, exist_ok=True)

    pages = convert_from_bytes(file_bytes, dpi=PDF_DPI)
    total_pages = len(pages)
    added, skipped = 0, 0

    for i, page in enumerate(pages):
        page_num = i + 1
        doc_id = f"{doc_id_prefix}__page_{page_num}"
        image_path = os.path.join(doc_image_folder, f"page_{page_num}.png")

        if collection.get(ids=[doc_id])["ids"]:
            skipped += 1
        else:
            rgb_page = page.convert("RGB")
            if not os.path.exists(image_path):
                rgb_page.save(image_path, "PNG")

            embedding = embed_page_image(rgb_page)
            collection.add(
                ids=[doc_id],
                embeddings=[embedding.tolist()],
                metadatas=[{
                    "page_number": page_num,
                    "image_path": image_path,
                    "filename": uploaded_file.name,
                    "doc_name": doc_name,
                    "doc_id_prefix": doc_id_prefix,
                }],
                documents=[f"{doc_name} — page {page_num}"],
            )
            added += 1

        if progress_bar is not None:
            progress_bar.progress(
                (i + 1) / total_pages,
                text=f"Indexing '{uploaded_file.name}' — page {page_num}/{total_pages}",
            )

    return {
        "doc_name": doc_name,
        "doc_id_prefix": doc_id_prefix,
        "total_pages": total_pages,
        "added": added,
        "skipped": skipped,
    }


def get_indexed_documents() -> dict:
    """Return {(doc_name, doc_id_prefix): page_count} for everything currently
    stored in ChromaDB — including Phase 2's original 26-page index."""
    if collection.count() == 0:
        return {}
    all_meta = collection.get(include=["metadatas"])["metadatas"]
    docs = {}
    for m in all_meta:
        key = (m.get("doc_name", "unknown"), m.get("doc_id_prefix", "unknown"))
        docs[key] = docs.get(key, 0) + 1
    return dict(sorted(docs.items(), key=lambda kv: kv[0][0].lower()))


def delete_document(doc_id_prefix: str) -> int:
    """Remove every page belonging to one document from the index."""
    matches = collection.get(where={"doc_id_prefix": doc_id_prefix})
    ids_to_delete = matches["ids"]
    if ids_to_delete:
        collection.delete(ids=ids_to_delete)
    folder = os.path.join(IMAGE_FOLDER, doc_id_prefix)
    if os.path.isdir(folder):
        shutil.rmtree(folder, ignore_errors=True)
    return len(ids_to_delete)


def reset_index():
    """Wipe the entire collection — used by the 'Clear all documents' button."""
    ids = collection.get()["ids"]
    if ids:
        collection.delete(ids=ids)
    if os.path.isdir(IMAGE_FOLDER):
        shutil.rmtree(IMAGE_FOLDER, ignore_errors=True)
        os.makedirs(IMAGE_FOLDER, exist_ok=True)


# ============================================================================
# Retrieval + generation -- Steps 5/7 logic, extended with an optional doc filter
# ============================================================================
def retrieve_top_k_pages(query_text: str, k: int = 3, doc_filter=None):
    query_embedding = embed_query(query_text)
    where = {"doc_id_prefix": {"$in": doc_filter}} if doc_filter else None

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k,
        where=where,
        include=["metadatas", "distances"],
    )

    pages = []
    if results["ids"] and results["ids"][0]:
        for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
            pages.append({
                "page_number": meta["page_number"],
                "image_path": meta["image_path"],
                "doc_name": meta["doc_name"],
                "similarity": 1 - dist,   # ChromaDB returns cosine distance
            })
    return pages


def build_multimodal_input(query_text: str, pages: list):
    content = [{"type": "text", "text": f"Question: {query_text}"}]
    for p in pages:
        with open(p["image_path"], "rb") as f:
            image_bytes = f.read()
        content.append({
            "type": "image",
            "data": base64.b64encode(image_bytes).decode("utf-8"),
            "mime_type": "image/png",
        })
    return content


def generate_answer(gemini_client, query_text: str, pages: list) -> str:
    try:
        interaction = gemini_client.interactions.create(
            model=GEMINI_MODEL,
            input=build_multimodal_input(query_text, pages),
            system_instruction=SYSTEM_INSTRUCTION,
        )
        return interaction.output_text
    except Exception as e:
        return f"Gemini request failed: {e}"


# ============================================================================
# Session state
# ============================================================================
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []        # [{query, answer, pages}, ...]
if "processed_uploads" not in st.session_state:
    st.session_state.processed_uploads = set()  # "<name>_<size>" already indexed this session


# ============================================================================
# Sidebar — Document Library (the new multi-PDF control panel)
# ============================================================================
st.sidebar.title("📁 Document Library")

if migrated_count:
    st.sidebar.caption(f"Migrated {migrated_count} legacy record(s) from Phase 2.")

if shutil.which("pdftoppm") is None:
    st.sidebar.error(
        "poppler-utils not found on this system.\n\n"
        "Install it before uploading PDFs:\n"
        "`apt-get install -y poppler-utils` (Linux/Colab)\n"
        "`brew install poppler` (macOS)"
    )

uploaded_files = st.sidebar.file_uploader(
    "Add PDFs to the knowledge base",
    type=["pdf"],
    accept_multiple_files=True,
    help="Each page is rendered as an image and embedded directly with ColPali — no OCR, any number of files.",
)

if uploaded_files:
    for uf in uploaded_files:
        upload_key = f"{uf.name}_{uf.size}"
        if upload_key in st.session_state.processed_uploads:
            continue  # already handled in this session — avoids re-embedding on every rerun

        progress_bar = st.sidebar.progress(0.0, text=f"Preparing '{uf.name}'...")
        try:
            result = index_pdf(uf, progress_bar=progress_bar)
            progress_bar.progress(
                1.0,
                text=f"✅ {uf.name}: {result['added']} new pages, {result['skipped']} already indexed",
            )
        except Exception as e:
            progress_bar.empty()
            st.sidebar.error(f"Failed to index '{uf.name}': {e}")
        finally:
            st.session_state.processed_uploads.add(upload_key)
        time.sleep(0.3)

st.sidebar.divider()
st.sidebar.subheader("Indexed Documents")

docs = get_indexed_documents()
if not docs:
    st.sidebar.caption("No documents indexed yet — upload one or more PDFs above.")
else:
    for (doc_name, doc_id_prefix), page_count in docs.items():
        row = st.sidebar.columns([4, 1])
        row[0].markdown(f"**{doc_name}**  \n{page_count} pages indexed")
        if row[1].button("🗑️", key=f"del_{doc_id_prefix}", help=f"Remove '{doc_name}'"):
            removed = delete_document(doc_id_prefix)
            st.sidebar.success(f"Removed {removed} pages from '{doc_name}'.")
            st.rerun()

    if st.sidebar.button("Clear ALL documents", type="secondary"):
        reset_index()
        st.session_state.chat_history = []
        st.rerun()

st.sidebar.divider()
st.sidebar.subheader("⚙️ Settings")

api_key = st.sidebar.text_input(
    "Gemini API key", type="password",
    value=os.environ.get("GEMINI_API_KEY", ""),
    help="Get one at aistudio.google.com/apikey",
)
top_k = st.sidebar.slider("Pages to retrieve (k)", min_value=1, max_value=8, value=3)

doc_names = [d[0] for d in docs.keys()]
selected_docs = st.sidebar.multiselect(
    "Search only in (optional)",
    options=doc_names,
    default=[],
    help="Leave empty to search across every indexed document.",
)
doc_filter = (
    [dip for (dn, dip) in docs.keys() if dn in selected_docs]
    if selected_docs else None
)

if st.sidebar.button("Clear chat history"):
    st.session_state.chat_history = []
    st.rerun()

st.sidebar.divider()
st.sidebar.caption(f"Total pages indexed: {collection.count()} · Device: {DEVICE}")


# ============================================================================
# Main area
# ============================================================================
st.title("📄 Visual-Doc Assistant")
st.caption("Multimodal RAG over your own documents — ColPali page-image retrieval + Gemini generation. No OCR.")

if not docs:
    st.info("👈 Upload one or more PDFs from the sidebar to get started. Any number of documents can be added, at any time.")

# Render prior turns first, in chronological order (standard chat layout)
for turn in st.session_state.chat_history:
    with st.chat_message("user"):
        st.markdown(turn["query"])
    with st.chat_message("assistant"):
        st.markdown(turn["answer"])
        with st.expander(f"📑 Source pages ({len(turn['pages'])})", expanded=False):
            cols = st.columns(min(len(turn["pages"]), 4) or 1)
            for i, p in enumerate(turn["pages"]):
                with cols[i % len(cols)]:
                    img = Image.open(p["image_path"]).convert("RGB")
                    st.image(
                        img,
                        caption=f"{p['doc_name']} · p.{p['page_number']} · sim {p['similarity']:.3f}",
                        use_container_width=True,
                    )

query = st.chat_input("Ask a question about your documents...")

if query:
    if not api_key:
        st.error("Please enter your Gemini API key in the sidebar.")
    elif not docs:
        st.warning("Please upload at least one PDF first.")
    else:
        with st.chat_message("user"):
            st.markdown(query)

        gemini_client = genai.Client(api_key=api_key)
        with st.chat_message("assistant"):
            with st.spinner("Retrieving relevant pages..."):
                pages = retrieve_top_k_pages(query, k=top_k, doc_filter=doc_filter)

            if not pages:
                st.warning("No relevant pages found in the selected document(s).")
            else:
                with st.spinner("Generating answer with Gemini..."):
                    answer = generate_answer(gemini_client, query, pages)

                st.markdown(answer)
                with st.expander(f"📑 Source pages ({len(pages)})", expanded=True):
                    cols = st.columns(min(len(pages), 4) or 1)
                    for i, p in enumerate(pages):
                        with cols[i % len(cols)]:
                            img = Image.open(p["image_path"]).convert("RGB")
                            st.image(
                                img,
                                caption=f"{p['doc_name']} · p.{p['page_number']} · sim {p['similarity']:.3f}",
                                use_container_width=True,
                            )

                st.session_state.chat_history.append({
                    "query": query, "answer": answer, "pages": pages,
                })


Overwriting /content/drive/MyDrive/Visual-Doc/app.py


## Step 9D — True Multi-Vector Late-Interaction (MaxSim) Retrieval

Every step so far (9, 9B) stored **one mean-pooled 128-dim vector per page**
and retrieved with ChromaDB's plain cosine search — a deliberate
simplification from what the ColPali paper actually scores:

$$LI(q, d) = \sum_i \max_j \langle E_q(i) \mid E_d(j) \rangle$$

For each query token, find its single best-matching document patch, then
sum those best-match scores. Mean-pooling collapses every patch on a page
into one average vector *before* any comparison happens — a page can
average out to "close enough" while no single patch on it actually answers
the question, and a page with one perfectly-matching patch buried among
unrelated ones can lose to a page that's merely uniformly mediocre. This
cell implements the paper's real scoring function.

### Why this needs new storage, not just a new query

ChromaDB's ANN index is built for *one fixed-size vector per record* — that's
what makes its cosine search fast. MaxSim needs the *full patch matrix*
(~1024 × 128 per page) available to score against, which an ANN index can't
hold or search directly. So this cell:

1. Keeps storing the mean-pooled vector in ChromaDB, but now **only** as a
   cheap candidate-shortlist mechanism — not as the final score.
2. **Also** saves each page's full patch matrix to disk as a float16 `.npy`
   file (~256 KB/page, matching the paper's own reported storage footprint,
   Table 4) next to its PNG, with the path stored in ChromaDB metadata.
3. At query time, re-ranks the shortlist by computing the **exact** MaxSim
   formula above against those saved patch matrices.

This is the standard two-stage pattern real ColPali deployments use — the
paper itself points to PLAID/Vespa for this at Wikipedia scale; here it's
the same shortlist-then-exact-rerank idea built on ChromaDB + local files
instead of a dedicated multi-vector engine. An **Exhaustive** mode is also
included that skips the ANN shortlist entirely and scores every indexed
page exactly, for smaller libraries where zero approximation is preferred.

Indexing now also has to compute and save a much bigger per-page tensor, so
this cell folds in Step 9C's batched-embedding and parallel-rendering
optimizations too (if you haven't run a separate 9C, this is a strict
superset; if you have, this simply overwrites `app.py` the same way every
step here does).

**Everything else is unchanged:** the Streamlit UI shape, PDF upload flow,
Document Library, chat history, Gemini prompt construction
(`build_multimodal_input`), generation (`generate_answer`), and
`SYSTEM_INSTRUCTION` are identical to Step 9B. Existing documents from
Steps 9/9B are **not invalidated** — they still retrieve correctly via a
labeled fallback, and a one-click **"⏫ Upgrade to MaxSim"** button in the
Document Library sidebar backfills their full patch embeddings from the
page images already saved on disk — no re-upload of the original PDF
needed.

> **Sanity check before trusting this:** a synthetic test (two documents —
> one with a single precisely-matching patch buried in noise, one with
> broad-but-shallow relevance everywhere) confirms mean-pooling and MaxSim
> *disagree* on this pair, and MaxSim correctly favors the document with
> the precise match (score 5.54 vs 1.76, where mean-pooling had ranked it
> *behind* the diffuse document, 0.21 vs 0.72). That's exactly the failure
> mode late interaction exists to fix.


In [16]:
%%writefile /content/drive/MyDrive/Visual-Doc/app.py
"""
Visual-Doc Assistant — Phase 3, Step 9D: True Multi-Vector Late-Interaction (MaxSim)
======================================================================================
Every prior step (9, 9B, 9C) stored ONE mean-pooled 128-dim vector per page and
retrieved with ChromaDB's plain cosine search — a deliberate simplification from
the ColPali paper's actual scoring function, which keeps every patch/token vector
and computes:

    LI(q, d) = sum_i  max_j  <Eq(i) | Ed(j)>          (Eq. 1, ColPali paper)

i.e. for each query token, find its single best-matching document patch, then sum
those best-match scores. Mean-pooling collapses all of a page's patches into one
average vector BEFORE any comparison happens — a page can average out to "close
enough" to a query while no single patch on it actually answers the question, and
conversely a page with one perfectly-matching patch buried among unrelated ones
can lose to a page that's merely uniformly-mediocre. This cell implements the
paper's real scoring function.

WHY THIS NEEDS NEW STORAGE, NOT JUST A NEW QUERY
ChromaDB's ANN index is built for ONE fixed-size vector per record — that's what
makes its cosine search fast. MaxSim needs the FULL patch matrix (~1024 x 128 per
page) available to score against, which isn't something an ANN index can hold or
search directly. So this cell:

  1. Keeps storing the mean-pooled vector in ChromaDB, but now ONLY as a cheap
     candidate-shortlist mechanism (fast, approximate) — not as the final score.
  2. ALSO saves each page's full patch matrix to disk as a float16 .npy file
     (~256 KB/page, matching the paper's own reported storage footprint, Table 4)
     next to its PNG, path stored in ChromaDB metadata.
  3. At query time, re-ranks the shortlist by computing the EXACT MaxSim formula
     above against those saved patch matrices — this is the real ColPali score,
     not an approximation.

This is the standard two-stage pattern real ColPali deployments use (the paper
itself points to PLAID/Vespa for this at Wikipedia-scale); we're doing the same
shortlist-then-exact-rerank idea with ChromaDB + local files instead of a
dedicated multi-vector engine. An "Exhaustive" mode is also included that skips
the ANN shortlist entirely and scores every indexed page exactly, for anyone who
wants zero approximation on a smaller library.

Since indexing now also has to compute and save a much bigger per-page tensor
(not just one small vector), this cell folds in Step 9C's batched-embedding and
parallel-rendering optimizations too — without them, MaxSim indexing would be
slower than 9B's single-page loop ever was. If you already applied 9C
separately, this is a strict superset and simply overwrites app.py the same way
every step here does.

EVERYTHING ELSE IS UNCHANGED: the Streamlit UI shape, PDF upload flow, Document
Library, chat history, Gemini prompt construction (build_multimodal_input),
generation (generate_answer), and SYSTEM_INSTRUCTION are identical to Step 9B.
Existing documents from Steps 9/9B/9C are NOT invalidated — they still retrieve
correctly using the fallback described below, and a one-click "Upgrade to
MaxSim" button (Document Library sidebar) backfills their full patch embeddings
from the page images already saved on disk — no re-upload of the original PDF
needed.
"""
import os
import shutil
import base64
import hashlib
import time

import numpy as np
import streamlit as st
import torch
from PIL import Image
import chromadb
from pdf2image import convert_from_bytes, pdfinfo_from_bytes
from colpali_engine.models import ColPali, ColPaliProcessor
from google import genai

# ============================================================================
# Config
# ============================================================================
MODEL_NAME       = "vidore/colpali-v1.2"
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_FOLDER     = "/content/drive/MyDrive/Visual-Doc/processed_images"
DB_PATH          = "/content/drive/MyDrive/Visual-Doc/chroma_db_storage"

GEMINI_MODEL     = "gemini-3.6-flash"
DEFAULT_DPI      = 200
DEFAULT_BATCH    = 4
RENDER_THREADS   = max(1, os.cpu_count() or 4)
PATCH_DTYPE      = np.float16   # matches the paper's own storage footprint (Table 4)
DEFAULT_SHORTLIST = 20

os.makedirs(IMAGE_FOLDER, exist_ok=True)
os.makedirs(DB_PATH, exist_ok=True)

SYSTEM_INSTRUCTION = (
    "You are the Visual-Doc Assistant, a technical documentation Q&A system. "
    "Answer the user's question using ONLY the page image(s) provided below — "
    "do not use outside knowledge. Reference the source document and page "
    "number when relevant. If the provided pages don't contain the answer, "
    "say so directly instead of guessing."
)

st.set_page_config(page_title="Visual-Doc Assistant", page_icon="📄", layout="wide")


# ============================================================================
# Cached resources
# ============================================================================
@st.cache_resource(show_spinner="Loading ColPali model (first load only)...")
def load_colpali():
    model = ColPali.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda:0"
    ).eval()
    processor = ColPaliProcessor.from_pretrained(MODEL_NAME)
    return model, processor


@st.cache_resource(show_spinner="Connecting to ChromaDB...")
def load_collection():
    db_client = chromadb.PersistentClient(path=DB_PATH)
    return db_client.get_or_create_collection(
        name="visual_doc_collection", metadata={"hnsw:space": "cosine"}
    )


model, processor = load_colpali()
collection = load_collection()


@st.cache_resource(show_spinner="Checking index for legacy records...")
def migrate_legacy_metadata():
    """One-time backfill for pre-Step-9B records missing doc_id_prefix.
    Unrelated to MaxSim — unchanged from Step 9B."""
    if collection.count() == 0:
        return 0
    all_data = collection.get(include=["metadatas"])
    updates_ids, updates_meta = [], []
    for doc_id, meta in zip(all_data["ids"], all_data["metadatas"]):
        if "doc_id_prefix" not in meta:
            legacy_prefix = f"{meta.get('doc_name', 'legacy')}__legacy"
            new_meta = dict(meta)
            new_meta["doc_id_prefix"] = legacy_prefix
            updates_ids.append(doc_id)
            updates_meta.append(new_meta)
    if updates_ids:
        collection.update(ids=updates_ids, metadatas=updates_meta)
    return len(updates_ids)


migrated_count = migrate_legacy_metadata()


# ============================================================================
# Embedding functions — now return TRUE multi-vector representations
# ============================================================================
@torch.no_grad()
def embed_page_images_batch(images: list):
    """
    Returns a list of (full, proxy) per image:
      full  — (N_patches, 128) float16 — the TRUE multi-vector patch embedding,
              saved to disk and used for exact MaxSim scoring.
      proxy — (128,) float32 — mean-pooled, kept ONLY as a fast ANN
              candidate-shortlist key (same role Steps 9B/9C used it for
              directly as the final score — it no longer is one).
    """
    inputs = processor.process_images(images).to(DEVICE)
    batch_patch_embeddings = model(**inputs)              # (B, N_patches, 128)
    results = []
    for patch_emb in batch_patch_embeddings:
        full32 = patch_emb.cpu().float().numpy()
        full = full32.astype(PATCH_DTYPE)                  # (N_patches, 128)
        proxy = full32.mean(axis=0).astype(np.float32)      # (128,)
        results.append((full, proxy))
    return results


@torch.no_grad()
def embed_query(query_text: str):
    """
    Returns (proxy, full_tokens):
      proxy       — (128,) mean-pooled, for ANN shortlisting only.
      full_tokens — (N_tokens, 128), the TRUE per-token query embedding used
                    for exact MaxSim scoring (includes ColBERT-style query
                    augmentation tokens, applied automatically by
                    ColPaliProcessor — unchanged from every prior phase).
    """
    inputs = processor.process_queries([query_text]).to(DEVICE)
    token_embeddings = model(**inputs)                      # (1, N_tokens, 128)
    full_tokens = token_embeddings.squeeze(0).cpu().float().numpy()
    proxy = full_tokens.mean(axis=0)
    return proxy, full_tokens


def maxsim_score(query_tokens: np.ndarray, doc_patches: np.ndarray) -> float:
    """
    Exact ColPali/ColBERT late interaction:  LI(q,d) = sum_i max_j <Eq(i)|Ed(j)>
    query_tokens: (Nq, 128)   doc_patches: (Nd, 128)   No padding needed —
    each document is scored independently, so patch counts can differ freely
    between pages.
    """
    sims = query_tokens.astype(np.float32) @ doc_patches.astype(np.float32).T   # (Nq, Nd)
    return float(sims.max(axis=1).sum())


def _cosine(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-8
    return float(np.dot(a, b) / denom)


@st.cache_data(show_spinner=False)
def _load_patch_embeddings(path: str) -> np.ndarray:
    return np.load(path).astype(np.float32)


# ============================================================================
# PDF ingestion pipeline — Step 9C's speed fixes, extended to save full patches
# ============================================================================
def _file_hash(file_bytes: bytes) -> str:
    return hashlib.md5(file_bytes).hexdigest()[:10]


def index_pdf(uploaded_file, dpi: int, batch_size: int, progress_bar=None) -> dict:
    file_bytes = uploaded_file.getvalue()
    doc_hash = _file_hash(file_bytes)
    doc_name = os.path.splitext(uploaded_file.name)[0]
    doc_id_prefix = f"{doc_name}__{doc_hash}"
    doc_image_folder = os.path.join(IMAGE_FOLDER, doc_id_prefix)

    t_start = time.time()

    try:
        total_pages = pdfinfo_from_bytes(file_bytes)["Pages"]
    except Exception:
        total_pages = None

    if total_pages is not None:
        already = collection.get(where={"doc_id_prefix": doc_id_prefix}, include=[])
        if len(already["ids"]) >= total_pages:
            if progress_bar is not None:
                progress_bar.progress(1.0, text=f"'{uploaded_file.name}' already fully indexed — skipped")
            return {
                "doc_name": doc_name, "doc_id_prefix": doc_id_prefix,
                "total_pages": total_pages, "added": 0, "skipped": total_pages,
                "elapsed": time.time() - t_start,
            }

    os.makedirs(doc_image_folder, exist_ok=True)

    pages = convert_from_bytes(file_bytes, dpi=dpi, thread_count=RENDER_THREADS)
    total_pages = len(pages)

    candidate_ids = [f"{doc_id_prefix}__page_{i + 1}" for i in range(total_pages)]
    existing_ids = set(collection.get(ids=candidate_ids, include=[])["ids"])

    to_process = [
        (i + 1, page) for i, page in enumerate(pages)
        if f"{doc_id_prefix}__page_{i + 1}" not in existing_ids
    ]
    skipped = total_pages - len(to_process)
    added = 0

    for batch_start in range(0, len(to_process), batch_size):
        batch = to_process[batch_start: batch_start + batch_size]
        batch_page_nums = [pn for pn, _ in batch]
        batch_images_rgb = [pg.convert("RGB") for _, pg in batch]

        batch_paths = []
        for page_num, rgb_img in zip(batch_page_nums, batch_images_rgb):
            image_path = os.path.join(doc_image_folder, f"page_{page_num}.png")
            if not os.path.exists(image_path):
                rgb_img.save(image_path, "PNG")
            batch_paths.append(image_path)

        embedded = embed_page_images_batch(batch_images_rgb)   # [(full, proxy), ...]

        ids_batch, embs_batch, metas_batch, docs_batch = [], [], [], []
        for page_num, image_path, (full, proxy) in zip(batch_page_nums, batch_paths, embedded):
            patch_path = os.path.join(doc_image_folder, f"page_{page_num}_patches.npy")
            np.save(patch_path, full)

            ids_batch.append(f"{doc_id_prefix}__page_{page_num}")
            embs_batch.append(proxy.tolist())
            metas_batch.append({
                "page_number": page_num,
                "image_path": image_path,
                "filename": uploaded_file.name,
                "doc_name": doc_name,
                "doc_id_prefix": doc_id_prefix,
                "patch_embedding_path": patch_path,
                "num_patches": int(full.shape[0]),
            })
            docs_batch.append(f"{doc_name} — page {page_num}")

        collection.add(ids=ids_batch, embeddings=embs_batch, metadatas=metas_batch, documents=docs_batch)
        added += len(batch)

        if progress_bar is not None:
            done = batch_start + len(batch)
            progress_bar.progress(
                min(done / max(len(to_process), 1), 1.0),
                text=f"Embedding '{uploaded_file.name}' — {done}/{len(to_process)} new pages "
                     f"({skipped} already indexed)",
            )

    if progress_bar is not None:
        progress_bar.progress(1.0, text=f"'{uploaded_file.name}' done — {added} added, {skipped} skipped")

    return {
        "doc_name": doc_name, "doc_id_prefix": doc_id_prefix,
        "total_pages": total_pages, "added": added, "skipped": skipped,
        "elapsed": time.time() - t_start,
    }


def get_indexed_documents() -> dict:
    if collection.count() == 0:
        return {}
    all_meta = collection.get(include=["metadatas"])["metadatas"]
    docs = {}
    for m in all_meta:
        key = (m.get("doc_name", "unknown"), m.get("doc_id_prefix", "unknown"))
        docs[key] = docs.get(key, 0) + 1
    return dict(sorted(docs.items(), key=lambda kv: kv[0][0].lower()))


def get_maxsim_readiness() -> dict:
    """Return {(doc_name, doc_id_prefix): (pages_with_full_embeddings, total_pages)}."""
    if collection.count() == 0:
        return {}
    all_meta = collection.get(include=["metadatas"])["metadatas"]
    stats = {}
    for m in all_meta:
        key = (m.get("doc_name", "unknown"), m.get("doc_id_prefix", "unknown"))
        ready, total = stats.get(key, (0, 0))
        total += 1
        patch_path = m.get("patch_embedding_path")
        if patch_path and os.path.exists(patch_path):
            ready += 1
        stats[key] = (ready, total)
    return stats


def upgrade_document_to_maxsim(doc_id_prefix: str, batch_size: int, progress_bar=None) -> dict:
    """
    Backfill full patch embeddings for pages indexed before Step 9D (or an
    interrupted earlier upgrade). Reuses the page PNGs already saved on disk
    by Steps 9B/9C — the original PDF is NOT needed.
    """
    matches = collection.get(where={"doc_id_prefix": doc_id_prefix}, include=["metadatas"])
    to_upgrade = [
        (mid, meta) for mid, meta in zip(matches["ids"], matches["metadatas"])
        if not (meta.get("patch_embedding_path") and os.path.exists(meta.get("patch_embedding_path", "")))
    ]
    total = len(to_upgrade)
    upgraded = 0

    for batch_start in range(0, total, batch_size):
        batch = to_upgrade[batch_start: batch_start + batch_size]
        batch_ids = [mid for mid, _ in batch]
        batch_metas = [meta for _, meta in batch]
        batch_images = [Image.open(meta["image_path"]).convert("RGB") for meta in batch_metas]

        embedded = embed_page_images_batch(batch_images)

        new_metas, new_embs = [], []
        for meta, (full, proxy) in zip(batch_metas, embedded):
            folder = os.path.dirname(meta["image_path"])
            patch_path = os.path.join(folder, f"page_{meta['page_number']}_patches.npy")
            np.save(patch_path, full)
            m2 = dict(meta)
            m2["patch_embedding_path"] = patch_path
            m2["num_patches"] = int(full.shape[0])
            new_metas.append(m2)
            new_embs.append(proxy.tolist())

        collection.update(ids=batch_ids, embeddings=new_embs, metadatas=new_metas)
        upgraded += len(batch)

        if progress_bar is not None:
            progress_bar.progress(
                min(upgraded / max(total, 1), 1.0),
                text=f"Upgrading to MaxSim — {upgraded}/{total} pages",
            )

    if progress_bar is not None:
        progress_bar.progress(1.0, text=f"Upgrade complete — {upgraded} pages")

    return {"upgraded": upgraded, "total": total}


def delete_document(doc_id_prefix: str) -> int:
    matches = collection.get(where={"doc_id_prefix": doc_id_prefix})
    ids_to_delete = matches["ids"]
    if ids_to_delete:
        collection.delete(ids=ids_to_delete)
    folder = os.path.join(IMAGE_FOLDER, doc_id_prefix)
    if os.path.isdir(folder):
        shutil.rmtree(folder, ignore_errors=True)
    return len(ids_to_delete)


def reset_index():
    ids = collection.get()["ids"]
    if ids:
        collection.delete(ids=ids)
    if os.path.isdir(IMAGE_FOLDER):
        shutil.rmtree(IMAGE_FOLDER, ignore_errors=True)
        os.makedirs(IMAGE_FOLDER, exist_ok=True)


# ============================================================================
# Retrieval — two-stage: ANN shortlist (or exhaustive) + exact MaxSim rerank
# ============================================================================
def retrieve_top_k_pages(query_text: str, k: int = 3, doc_filter=None,
                          mode: str = "fast", shortlist_size: int = DEFAULT_SHORTLIST):
    """
    mode="fast" (default): ChromaDB's ANN index over the mean-pooled proxy
        vector produces a shortlist of `shortlist_size` candidates FAST, then
        every candidate is re-scored with the EXACT MaxSim formula and
        re-ranked — the score that determines the final top-k is always the
        true late-interaction score, never the proxy.
    mode="exhaustive": skips the ANN shortlist and MaxSim-scores every
        indexed page (respecting doc_filter) — exact by construction, no
        approximation anywhere, but scales linearly with library size.

    Pages indexed before Step 9D (no saved patch matrix yet) fall back to
    proxy-vector cosine similarity so they still participate in ranking,
    clearly labeled `score_type: "approx"` rather than silently mixed in
    as if it were a real MaxSim score.
    """
    proxy, query_tokens = embed_query(query_text)
    where = {"doc_id_prefix": {"$in": doc_filter}} if doc_filter else None

    if mode == "exhaustive":
        candidates = collection.get(where=where, include=["metadatas"])
        cand_ids, cand_metas = candidates["ids"], candidates["metadatas"]
    else:
        n_results = min(shortlist_size, max(collection.count(), 1))
        results = collection.query(
            query_embeddings=[proxy.tolist()], n_results=n_results,
            where=where, include=["metadatas"],
        )
        cand_ids = results["ids"][0] if results["ids"] else []
        cand_metas = results["metadatas"][0] if results["metadatas"] else []

    scored, legacy_idx = [], []
    for cid, meta in zip(cand_ids, cand_metas):
        patch_path = meta.get("patch_embedding_path")
        if patch_path and os.path.exists(patch_path):
            doc_patches = _load_patch_embeddings(patch_path)
            score = maxsim_score(query_tokens, doc_patches)
            scored.append({"id": cid, "meta": meta, "score": score, "score_type": "MaxSim"})
        else:
            scored.append({"id": cid, "meta": meta, "score": None, "score_type": "approx"})
            legacy_idx.append(len(scored) - 1)

    if legacy_idx:
        legacy_ids = [scored[i]["id"] for i in legacy_idx]
        fetched = collection.get(ids=legacy_ids, include=["embeddings"])
        emb_by_id = dict(zip(fetched["ids"], fetched["embeddings"]))
        for i in legacy_idx:
            doc_proxy = np.array(emb_by_id[scored[i]["id"]])
            scored[i]["score"] = _cosine(proxy, doc_proxy)

    scored.sort(key=lambda s: s["score"], reverse=True)

    pages = []
    for s in scored[:k]:
        meta = s["meta"]
        pages.append({
            "page_number": meta["page_number"],
            "image_path": meta["image_path"],
            "doc_name": meta["doc_name"],
            "score": s["score"],
            "score_type": s["score_type"],
        })
    return pages


def build_multimodal_input(query_text: str, pages: list):
    content = [{"type": "text", "text": f"Question: {query_text}"}]
    for p in pages:
        with open(p["image_path"], "rb") as f:
            image_bytes = f.read()
        content.append({
            "type": "image",
            "data": base64.b64encode(image_bytes).decode("utf-8"),
            "mime_type": "image/png",
        })
    return content


def generate_answer(gemini_client, query_text: str, pages: list) -> str:
    try:
        interaction = gemini_client.interactions.create(
            model=GEMINI_MODEL,
            input=build_multimodal_input(query_text, pages),
            system_instruction=SYSTEM_INSTRUCTION,
        )
        return interaction.output_text
    except Exception as e:
        return f"Gemini request failed: {e}"


# ============================================================================
# Session state
# ============================================================================
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "processed_uploads" not in st.session_state:
    st.session_state.processed_uploads = set()


# ============================================================================
# Sidebar — Document Library
# ============================================================================
st.sidebar.title("📁 Document Library")

if migrated_count:
    st.sidebar.caption(f"Migrated {migrated_count} legacy record(s) from Phase 2.")

if shutil.which("pdftoppm") is None:
    st.sidebar.error(
        "poppler-utils not found on this system.\n\n"
        "Install it before uploading PDFs:\n"
        "`apt-get install -y poppler-utils` (Linux/Colab)\n"
        "`brew install poppler` (macOS)"
    )

with st.sidebar.expander("⚡ Indexing speed settings", expanded=False):
    st.caption(
        "Lower DPI speeds up rasterization without hurting retrieval quality "
        "(ColPali resizes internally regardless) — it only affects how sharp "
        "the page looks when Gemini reads it."
    )
    render_dpi = st.slider("Rendering DPI", min_value=100, max_value=300, value=DEFAULT_DPI, step=25)
    embed_batch_size = st.slider(
        "Embedding batch size", min_value=1, max_value=8, value=DEFAULT_BATCH,
        help="Pages embedded per model forward pass. Lower this if you hit a CUDA out-of-memory error "
             "— storing full patch matrices now uses more VRAM per page than mean-pooling did.",
    )

with st.sidebar.expander("🎯 Retrieval settings", expanded=False):
    st.caption(
        "Fast mode uses the mean-pooled vector only to shortlist candidates via "
        "ChromaDB's ANN index, then re-ranks that shortlist with the exact "
        "ColBERT-style MaxSim score — the score that decides the final answer is "
        "always the real late-interaction score. Exhaustive mode skips the "
        "shortlist and scores every indexed page exactly."
    )
    retrieval_mode_label = st.radio(
        "Mode",
        ["Fast (ANN shortlist + exact rerank)", "Exhaustive (exact over full library)"],
        index=0,
    )
    retrieval_mode = "exhaustive" if retrieval_mode_label.startswith("Exhaustive") else "fast"
    shortlist_size = st.slider(
        "Shortlist size (Fast mode)", min_value=5, max_value=50, value=DEFAULT_SHORTLIST,
        disabled=(retrieval_mode == "exhaustive"),
    )
    if retrieval_mode == "exhaustive":
        st.caption(f"⚠️ Will exactly score all {collection.count()} indexed pages per query.")

uploaded_files = st.sidebar.file_uploader(
    "Add PDFs to the knowledge base",
    type=["pdf"],
    accept_multiple_files=True,
    help="Any number of files. Each page is rendered as an image and embedded directly with ColPali — no OCR.",
)

if uploaded_files:
    for uf in uploaded_files:
        upload_key = f"{uf.name}_{uf.size}"
        if upload_key in st.session_state.processed_uploads:
            continue

        progress_bar = st.sidebar.progress(0.0, text=f"Preparing '{uf.name}'...")
        try:
            result = index_pdf(uf, dpi=render_dpi, batch_size=embed_batch_size, progress_bar=progress_bar)
            progress_bar.progress(
                1.0,
                text=(f"✅ {uf.name}: {result['added']} new pages, {result['skipped']} already indexed "
                      f"({result['elapsed']:.1f}s)"),
            )
        except Exception as e:
            progress_bar.empty()
            st.sidebar.error(f"Failed to index '{uf.name}': {e}")
        finally:
            st.session_state.processed_uploads.add(upload_key)

st.sidebar.divider()
st.sidebar.subheader("Indexed Documents")

docs = get_indexed_documents()
readiness = get_maxsim_readiness()

if not docs:
    st.sidebar.caption("No documents indexed yet — upload one or more PDFs above.")
else:
    for (doc_name, doc_id_prefix), page_count in docs.items():
        ready, total = readiness.get((doc_name, doc_id_prefix), (0, page_count))
        status = "✅ MaxSim-ready" if ready == total else f"⏳ {ready}/{total} MaxSim-ready"

        row = st.sidebar.columns([4, 1])
        row[0].markdown(f"**{doc_name}**  \n{page_count} pages · {status}")
        if row[1].button("🗑️", key=f"del_{doc_id_prefix}", help=f"Remove '{doc_name}'"):
            removed = delete_document(doc_id_prefix)
            st.sidebar.success(f"Removed {removed} pages from '{doc_name}'.")
            st.rerun()

        if ready < total:
            if st.sidebar.button(f"⏫ Upgrade '{doc_name}' to MaxSim", key=f"upg_{doc_id_prefix}"):
                pb = st.sidebar.progress(0.0, text="Starting upgrade...")
                res = upgrade_document_to_maxsim(doc_id_prefix, batch_size=embed_batch_size, progress_bar=pb)
                st.sidebar.success(f"Upgraded {res['upgraded']} of {res['total']} pages.")
                st.rerun()

    if st.sidebar.button("Clear ALL documents", type="secondary"):
        reset_index()
        st.session_state.chat_history = []
        st.rerun()

st.sidebar.divider()
st.sidebar.subheader("⚙️ Settings")

api_key = st.sidebar.text_input(
    "Gemini API key", type="password",
    value=os.environ.get("GEMINI_API_KEY", ""),
    help="Get one at aistudio.google.com/apikey",
)
top_k = st.sidebar.slider("Pages to retrieve (k)", min_value=1, max_value=8, value=3)

doc_names = [d[0] for d in docs.keys()]
selected_docs = st.sidebar.multiselect(
    "Search only in (optional)",
    options=doc_names,
    default=[],
    help="Leave empty to search across every indexed document.",
)
doc_filter = (
    [dip for (dn, dip) in docs.keys() if dn in selected_docs]
    if selected_docs else None
)

if st.sidebar.button("Clear chat history"):
    st.session_state.chat_history = []
    st.rerun()

st.sidebar.divider()
st.sidebar.caption(f"Total pages indexed: {collection.count()} · Device: {DEVICE}")


# ============================================================================
# Main area
# ============================================================================
st.title("📄 Visual-Doc Assistant")
st.caption(
    "Multimodal RAG over your own documents — ColPali page-image retrieval with "
    "true multi-vector late-interaction (MaxSim) + Gemini generation. No OCR."
)

if not docs:
    st.info("👈 Upload one or more PDFs from the sidebar to get started. Any number of documents can be added, at any time.")


def _score_caption(p):
    label = "MaxSim" if p["score_type"] == "MaxSim" else "≈cos"
    return f"{p['doc_name']} · p.{p['page_number']} · {label} {p['score']:.3f}"


for turn in st.session_state.chat_history:
    with st.chat_message("user"):
        st.markdown(turn["query"])
    with st.chat_message("assistant"):
        st.markdown(turn["answer"])
        with st.expander(f"📑 Source pages ({len(turn['pages'])})", expanded=False):
            cols = st.columns(min(len(turn["pages"]), 4) or 1)
            for i, p in enumerate(turn["pages"]):
                with cols[i % len(cols)]:
                    img = Image.open(p["image_path"]).convert("RGB")
                    st.image(img, caption=_score_caption(p), use_container_width=True)

query = st.chat_input("Ask a question about your documents...")

if query:
    if not api_key:
        st.error("Please enter your Gemini API key in the sidebar.")
    elif not docs:
        st.warning("Please upload at least one PDF first.")
    else:
        with st.chat_message("user"):
            st.markdown(query)

        gemini_client = genai.Client(api_key=api_key)
        with st.chat_message("assistant"):
            with st.spinner("Retrieving relevant pages (MaxSim)..."):
                pages = retrieve_top_k_pages(
                    query, k=top_k, doc_filter=doc_filter,
                    mode=retrieval_mode, shortlist_size=shortlist_size,
                )

            if not pages:
                st.warning("No relevant pages found in the selected document(s).")
            else:
                with st.spinner("Generating answer with Gemini..."):
                    answer = generate_answer(gemini_client, query, pages)

                st.markdown(answer)
                with st.expander(f"📑 Source pages ({len(pages)})", expanded=True):
                    cols = st.columns(min(len(pages), 4) or 1)
                    for i, p in enumerate(pages):
                        with cols[i % len(cols)]:
                            img = Image.open(p["image_path"]).convert("RGB")
                            st.image(img, caption=_score_caption(p), use_container_width=True)

                st.session_state.chat_history.append({
                    "query": query, "answer": answer, "pages": pages,
                })


Overwriting /content/drive/MyDrive/Visual-Doc/app.py


## Step 10 — Launch the App from Colab

Launches whichever `app.py` was written last at `/content/drive/MyDrive/Visual-Doc/app.py` — if you ran Step 9B above, that's the multi-PDF version; if you skipped it, this launches Step 9's single-document version instead. No changes needed here either way.

Colab can't expose `localhost` directly, so `localtunnel` forwards port 8501 to a public URL. This runs the Streamlit process in the background and prints a tunnel link — open it, and when prompted for a "Tunnel Password," paste the IP printed below.

In [17]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!mv cloudflared /usr/local/bin/cloudflared

!pip install -q streamlit

!pkill -9 -f streamlit || true
!pkill -9 -f cloudflared || true

import time

!streamlit run /content/drive/MyDrive/Visual-Doc/app.py &>/content/logs.txt &
time.sleep(6)

!cloudflared tunnel --url http://localhost:8501 &>/content/cloudflared_logs.txt &
time.sleep(8)

!grep -o 'https://[a-zA-Z0-9.-]*\.trycloudflare\.com' /content/cloudflared_logs.txt | head -1

^C
^C
https://fat-quest-backgrounds-intensive.trycloudflare.com


---
## Phase 3 Summary

| Component | Detail |
|---|---|
| Retrieval | `embed_query()` (from Phase 2) → ChromaDB cosine search → Top-K pages |
| Generation model | Gemini 3.6 Flash (`gemini-3.6-flash`) via the `google-genai` Interactions API |
| Prompt construction | Query text + inline base64 page image(s) (PNG) + a grounding system instruction |
| Answer grounding | System instruction restricts Gemini to the retrieved page images only |
| UI | Streamlit app (`app.py`) — query box, Ask button, answer + source-page thumbnails |
| Deployment | Runs inside Colab (GPU needed for ColPali), exposed via localtunnel |

### Step 9B addendum
- **Uploading new PDFs through the UI is now implemented.** Step 9B rewrites `app.py` to wire Phase 1's PDF→image step and Phase 2's embedding loop directly into the Streamlit page — any number of documents, added and removed from the sidebar, no notebook re-runs required. Step 9's original single-document `app.py` is left above unmodified for reference.

### Step 9D addendum
- **Full late-interaction (MaxSim) retrieval is now implemented.** Step 9D saves each page's full patch matrix to disk (float16 `.npy`, ~256 KB/page) and re-ranks a ChromaDB-shortlisted candidate set with the exact ColBERT-style MaxSim formula — the mean-pooled vector from Steps 9B onward is now only a fast candidate-generation proxy, never the final score. An Exhaustive mode (exact MaxSim over the whole library, no shortlist) is available for smaller collections. Pre-9D documents keep working via a labeled cosine fallback and can be one-click upgraded to true MaxSim from the sidebar, reusing their already-saved page images.

### What's still not in scope
- **Quantitative retrieval evaluation.** No ViDoRe-style nDCG/Recall@K has been computed — verification remains qualitative (a synthetic mean-pooling-vs-MaxSim disagreement test confirming the scoring logic behaves as the paper describes, plus similarity/score values shown in the UI).

---
*Visual-Doc Assistant — NIT Patna, MCA (DS&I), Course: MC460502*